# Run Existing MCP Servers

MCP servers will do to LLMs what the app store did to mobile phones. Imagine doing things that you do online with an LLM - with MCP servers you can get your LLM to book a hotel or a flight and much more. Here we will use an MCP server to get coding information for your LLM.
In this project you will be introduced to using MCP servers, using transport protocols as a bridge from your application to the MCP server. We will introduce two transport protocols; the STDIO protocol, which will allow you to run MCP servers locally on your machine, and the HTTP protocol which will allow you to work with MCP servers hosted anywhere with internet access, as shown in the flowchart below.

<pre style="text-align: center;">
Client or Application  ←→  [transport]  ←→  MCP Server
   (LLM)           (Bridge)        (Local/Remote)
</pre>

Once you get the connection, the process is identical.


## STDIO INTRODUCTION

### STDIO Review
STDIO or Standard Input/Output is three communication channels that every program automatically gets. This was originally designed for Linux/Unix systems but is now used across all operating systems. This is for MCP servers that run locally.

### STDOUT 

```STDOUT``` (Standard Output) – The default stream where a program writes its normal output (usually your screen/terminal). We can print output using ```print()``` 


In [1]:
my_code="hello"
print(my_code)

hello


We can sketch the process. 

My Python Program: my_code = "hello" → print() →` [STDOUT] → Screen shows: hello


```sys.stdout.write()``` is a low-level (primitive) function that writes raw text directly to the standard output stream.

In [3]:
import sys
sys.stdout.write("Hello")

Hello

5

### STDIN

```STDIN``` (Standard Input) - Where input comes FROM (like your keyboard). Here we call the function input() with the prompt "What is your name?" You will then be prompted to input your name. The input from your keyboard will be stored in the variable name.


In [4]:
name = input("What is your name?")

sys.exit("Notebook exited after input.")

SystemExit: Notebook exited after input.

/home/tk-lpt-648/miniconda3/envs/mcp_env/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


We can sketch the process. We see that there are several steps:
<pre>
My Python Program → [STDOUT] → You (screen)
      input()      "What is your name?"
         ↓
You (keyboard) → [STDIN] → My Python Program  
    "Alice"                    stores in 'name'
         ↓
My Python Program → [STDERR] → You (screen)
     sys.exit()    "Notebook exited after input."
</pre>
We can see the varable ```name``` has the input value.


In [5]:
print(name)

Hasnain


### STDERR

```STDERR``` (standard error) is reserved for error messages and diagnostics. Both usually appear on the screen, but since they are separate streams, you can redirect them independently (e.g., send results to a file while keeping errors visible).


In [6]:
# Normal output goes to stdout
print("This is standard output (stdout)")

# Error messages can be written to stderr
print("This is an error message (stderr)", file=sys.stderr)

# Example: catching a real error
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Error occurred: {e}", file=sys.stderr)

This is standard output (stdout)


This is an error message (stderr)
Error occurred: division by zero


## Context7

In this lab we'll connect to the [Context7](https://context7.com/) MCP server via two different transport methods (STDIO and HTTP). Context7 is a service (with an MCP server) that provides up-to-date code documentation for LLMs and AI code editors. Once connected, we'll create `Client` instances, list the available tools, and call on those tools to get library documentation.

#### Input (what you give it):

- Library names (such as "fastmcp")
- Search queries for code documentation
- Requests for specific library information


#### Output (what it gives you back):

- Up-to-date code documentation formatted for LLMs
- Library information and compatibility details
- Code examples and usage patterns
- API references and function descriptions



### Importing Required Libraries

Run the following cell to import all required modules and dependencies for this lab. We'll need:
- `Client`: A communication outlet between an application (host) and an MCP server.
- `StdioTransport`: Communication run on subprocesses (read only input, write only output) for local MCP servers.
- `StreamableHttpTransport`: Communication for production deployments that run over bidirectional streaming on HTTP connections.


In [7]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport, StdioTransport

### STDIO Transport

For `StdioTransport`, everything runs on your system. STDIO Transport acts like a bridge from your Python file to an MCP server. Your notebook/Python file runs the `StdioTransport` code, your computer downloads the Context7 server code via npx, and your computer starts the Context7 server as a separate process. Both programs (your notebook + Context7 server) run on your machine, with STDIO Transport serving as the bridge (or wire) that connects your Python client to the Context7 server. This is the simplest way to connect to a local or CLI-packaged server as shown in the following diagram.

 
<pre style="text-align: center;">
Your Python Code  ←→  [stdio_transport]  ←→  Context7 Server on you com
    (Client)              (Bridge)              (your system)
</pre>



Let's create the transport.

**The `StdioTransport`:**
- **Launches** the Context7 server as a subprocess (`npx @upstash/context7-mcp`)


npx - A Node.js 
- Downloads the Context7 MCP server code from the npm registry
- Installs it temporarily (with -y flag to auto-confirm)
- Runs the MCP server program
- Starts listening on STDIN/STDOUT for MCP protocol messages

Python equivalent analogy: bash# 

Similar to doing this in Python world:
- pip install context7-mcp-server
- python -m context7-mcp-server

Why use npx?
- **Connects** your Python code's messages to the server's STDIN
- **Connects** the server's STDOUT back to your Python code
- **Manages** the communication flow between both sides


In [ ]:
stdio_transport = StdioTransport(
    command = "npx",
    args=["-y", "@upstash/context7-mcp"]
)
print(stdio_transport)

<StdioTransport(command='npx', args=['-y', '@upstash/context7-mcp'])>


## Client

Now let's wrap the transport in a client. Since the client needs a ```stdio_transport``` object, we defined the transport 'Client':


In [9]:
stdio_client = Client(stdio_transport)

The `Client` gives you a simple way to access the MCP server. Because communication with a server can take some time, Python uses asynchronous methods `async/await` to prevent the program from freezing while it waits for a response.

The line `async with stdio_client as client` uses Python's asynchronous context manager. This means that `Client` runs inside a special block that automatically takes care of starting and closing the connection, similar to how `with open(...)` works for files.


In [10]:
async with stdio_client as client:
    # List of tools the server provides
    tools = await client.list_tools()

print("Done")

Done


In [11]:
len(tools)

2

In [12]:
print(tools[0].name) # Name of 1st tool

resolve-library-id


In [13]:
print(tools[0].description) # Description of first tool

Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.

You MUST call this function before 'Query Documentation' tool to obtain a valid Context7-compatible library ID UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.

Each result includes:
- Library ID: Context7-compatible identifier (format: /org/project)
- Name: Library or package name
- Description: Short summary
- Code Snippets: Number of available code examples
- Source Reputation: Authority indicator (High, Medium, Low, or Unknown)
- Benchmark Score: Quality indicator (100 is the highest score)
- Versions: List of versions if available. Use one of those versions if the user provides a version in their query. The format of the version is /org/project/version.

For best results, select libraries based on name match, source reputation, snippet coverage, benchmark score, and relevance to your use case.

Selection Process

In [14]:
tools[0].inputSchema # Required parameters to call 1st tool

{'$schema': 'http://json-schema.org/draft-07/schema#',
 'type': 'object',
 'properties': {'query': {'type': 'string',
   'description': 'The question or task you need help with. This is used to rank library results by relevance to what the user is trying to accomplish. The query is sent to the Context7 API for processing. Do not include any sensitive or confidential information such as API keys, passwords, credentials, personal data, or proprietary code in your query.'},
  'libraryName': {'type': 'string',
   'description': "Library name to search for and retrieve a Context7-compatible library ID. Use the official library name with proper punctuation — e.g., 'Next.js' instead of 'nextjs', 'Customer.io' instead of 'customerio', 'Three.js' instead of 'threejs'."}},
 'required': ['query', 'libraryName']}

In [15]:
print(
f""" name: {tools[1].name}: \n
description: {tools[1].description} \n
inputSchema: {tools[1].inputSchema}""")

 name: query-docs: 

description: Retrieves and queries up-to-date documentation and code examples from Context7 for any programming library or framework.

You must call 'Resolve Context7 Library ID' tool first to obtain the exact Context7-compatible library ID required to use this tool, UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.

IMPORTANT: Do not call this tool more than 3 times per question. If you cannot find what you need after 3 calls, use the best information you have. 

inputSchema: {'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'libraryId': {'type': 'string', 'description': "Exact Context7-compatible library ID (e.g., '/mongodb/docs', '/vercel/next.js', '/supabase/supabase', '/vercel/next.js/v14.3.0-canary.87') retrieved from 'resolve-library-id' or directly from user query in the format '/org/project' or '/org/project/version'."}, 'query': {'type': 'string', '

## call_tool

`call_tool` is the method you use to actually run/execute a specific tool on the MCP server. 

The pramaters are:

- **`tool_name`** - The tool name comes from `.name` like `tools[0].name`, which gives you "resolve-library-id"

- **`{parameters}`** - The inputs the tool needs (like `{"libraryName": "fastmcp", "query":"[YOUR QUERY]"}`). The input structure comes from `.inputSchema`

- **`response`** - What the tool sends back to you


### resolve-library-id

Let's get information about the FastMCP library. We need the library identification, so we'll use the `resolve-library-id` tool. This will allow us to get the library information. The input parameter will be libraryName: "fastmcp".

In [16]:
async with stdio_client as client:
    # Find a library ID via a search query
    response = await client.call_tool("resolve-library-id", {
        "libraryName": "fastmcp",
        "query": "I want to create a new MCP server using the fastmcp Python framework"
    })

print(response.content[0].text)

Available Libraries:

- Title: FastMCP
- Context7-compatible library ID: /prefecthq/fastmcp
- Description: FastMCP is a Python framework for building Model Context Protocol (MCP) servers, clients, and apps that connect LLMs to tools, resources, and data with automatic schema generation and protocol management.
- Code Snippets: 2785
- Source Reputation: High
- Benchmark Score: 91.9
- Versions: v3.2.0
----------
- Title: FastMCP
- Context7-compatible library ID: /llmstxt/gofastmcp_llms_txt
- Description: FastMCP is a fast, Pythonic framework for building and deploying Multi-modal Conversational Platform (MCP) servers and clients, offering programmatic interfaces and extensive integrations for LLMs and various services.
- Code Snippets: 3577
- Source Reputation: High
- Benchmark Score: 83.64
----------
- Title: FastMCP
- Context7-compatible library ID: /websites/gofastmcp
- Description: FastMCP is a Python framework for building Model Context Protocol (MCP) servers and clients that connec

When you call the `resolve-library-id` tool, it returns a list of matching libraries for your search term. The output shows multiple options with details to help you choose the best one. Each result includes a Library ID (the unique identifier you'll need for the next step), a description of what the library does, the number of code snippets available, and a benchmark score indicating reliability.

Taking a look at the list, we'll choose the library with the highest **Benchmark Score**. The **Context7-compatible library ID** is `/prefecthq/fastmcp`.



### query-docs

Now we'll use the `query-docs` tool to retrieve the actual documentation for that specific library.

We'll use the context manager once again to fetch the code snippets and documentation of `/prefecthq/fastmcp` using the `query-docs` tool.


In [20]:
async with stdio_client as client:
    # Use resolved ID to fetch documentation
    docs = await client.call_tool("query-docs", {
        "libraryId": "/prefecthq/fastmcp",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000, # setting response limit of tool
    })

    print(docs.content[0].text[:1000])

### Consume Paginated Data with FastMCP Client

Source: https://github.com/prefecthq/fastmcp/blob/main/docs/servers/pagination.mdx

Demonstrates how to fetch paginated lists using the FastMCP client. Includes both automatic fetching for complete lists and manual iteration using cursors for memory-constrained environments.

```python
from fastmcp import Client

# Automatic pagination: fetches all pages automatically
async with Client(server) as client:
    tools = await client.list_tools()
    print(f"Total tools: {len(tools)}")

# Manual pagination: process results incrementally using cursors
async with Client(server) as client:
    result = await client.list_tools_mcp()
    print(f"Page 1: {len(result.tools)} tools")

    while result.nextCursor:
        result = await client.list_tools_mcp(cursor=result.nextCursor)
        print(f"Next page: {len(result.tools)} tools")
```

--------------------------------

### Retrieve tools from provider

Source: https://github.com/prefecthq/fastmc

## HTTP Preface

The HTTP protocol allows data to be transferred over the internet.

`Requests` is a Python Library that allows you to send `HTTP/1.1` requests easily. We can import the library as follows:


### HTTP Transport

Now let's talk to the Context7 MCP server over HTTPS—no local process needed. The HTTP transport also acts like a bridge, but instead of connecting from your notebook/Python file to your local computer, it connects to a remote server. This is ideal when the server is hosted remotely or you don't want to manage a subprocess. We'll use the same Client API with a different "wire" (transport). Once you create the transport, the process is pretty much identical.

<pre style="text-align: center;">
Your Python Code  ←→  [http_transport]  ←→  Context7 Server (remote)
   (Client)              (Bridge)              (Context7's servers)
</pre>

First, we create the `StreamableHttpTransport` object. 


In [ ]:
http_transport = StreamableHttpTransport(
    url="https://mcp.context7.com/mcp" # The server's MCP endpoint that exposes all the tools, resources and prompts
)
# streamable means: return response in streaming

In [19]:
http_client = Client(http_transport)

### List and call tools

In [21]:
async with http_client as client:
    tools = await client.list_tools()

    response = await client.call_tool("resolve-library-id", {
        "libraryName": "fastmcp",
        "query": "I want to create a new MCP server using the fastmcp Python framework"
    })

    docs = await client.call_tool("query-docs", {
        "libraryId": "/prefecthq/fastmcp",
        "query": "I want to fetch the code snippets and the documentation",
        "tokens": 5000
    })

In [23]:
for tool in tools:
        print(
f"""Tool Name: {tool.name}: \n
Tool Description: {tool.description} \n
Tool Input Schema: {tool.inputSchema}""")

Tool Name: resolve-library-id: 

Tool Description: Resolves a package/product name to a Context7-compatible library ID and returns matching libraries.

You MUST call this function before 'Query Documentation' tool to obtain a valid Context7-compatible library ID UNLESS the user explicitly provides a library ID in the format '/org/project' or '/org/project/version' in their query.

Each result includes:
- Library ID: Context7-compatible identifier (format: /org/project)
- Name: Library or package name
- Description: Short summary
- Code Snippets: Number of available code examples
- Source Reputation: Authority indicator (High, Medium, Low, or Unknown)
- Benchmark Score: Quality indicator (100 is the highest score)
- Versions: List of versions if available. Use one of those versions if the user provides a version in their query. The format of the version is /org/project/version.

For best results, select libraries based on name match, source reputation, snippet coverage, benchmark score,

In [26]:
print("Truncated response from 1st tool call: ", response.content[0].text[:500])
print("\n\nTruncated response from 2nd tool call: ", docs.content[0].text[:500]) 

Truncated response from 1st tool call:  Available Libraries:

- Title: FastMCP
- Context7-compatible library ID: /prefecthq/fastmcp
- Description: FastMCP is a Python framework for building Model Context Protocol (MCP) servers, clients, and apps that connect LLMs to tools, resources, and data with automatic schema generation and protocol management.
- Code Snippets: 2785
- Source Reputation: High
- Benchmark Score: 91.9
- Versions: v3.2.0
----------
- Title: FastMCP
- Context7-compatible library ID: /llmstxt/gofastmcp_llms_txt
- Descr


Truncated response from 2nd tool call:  ### Consume Paginated Data with FastMCP Client

Source: https://github.com/prefecthq/fastmcp/blob/main/docs/servers/pagination.mdx

Demonstrates how to fetch paginated lists using the FastMCP client. Includes both automatic fetching for complete lists and manual iteration using cursors for memory-constrained environments.

```python
from fastmcp import Client

# Automatic pagination: fetches all pages automaticall

---

## Other Transports

There are other transports but we won't demo them here as they are not as widely used as STDIO or HTTP. These two are enough for most use cases of MCP servers.

### SSE Transport

The Server-Sent Events transport enables HTTP communication between MCP servers and clients using an asymmetric channel. It uses SSE for server-to-client streaming and HTTP POST for client-to-server messages. It was replaced by the Streamable HTTP transport to provide bidirectional streaming capabilities and improved real-time communication efficiency.

Read more [here](https://gofastmcp.com/clients/transports#sse-transport-legacy).

### In-Memory Transport

In-memory transport connects a client directly to a FastMCP server instance within the same Python process. This eliminates network overhead by enabling direct function calls between client and server components. 

Read more [here](https://gofastmcp.com/clients/transports#in-memory-transport).
